# Encoding Categorical Data | Ordinal Encoding | Label Encoding

src: https://www.youtube.com/watch?v=w2GglmYHfmM&list=PLKnIA16_RmvYXWH_E6PuVLLHHTWXwwDN7&index=4

---

## 🎓 Encoding Ordinal and Nominal Categorical Variables Using ColumnTransformer

### 📌 Background Context

In machine learning workflows, it is essential to convert **categorical data** into **numeric format**, because models like **XGBoost**, **Logistic Regression**, and **Random Forest** cannot operate directly on string or symbolic categories.

From the earlier transcript, we understand:

* **Categorical features** may be either:

  * **Nominal** (no order, e.g., gender, state)
  * **Ordinal** (with order, e.g., education level, product review)

> In this demo, we will **ignore numerical features like `Age`** and **nominal features like `Gender`**, to **focus entirely on encoding categorical columns** using appropriate strategies.

---

### 🧠 Why Not Use a Single Encoder for All?

Using a single encoder (like `LabelEncoder`) for all features may lead to incorrect assumptions:

* Nominal categories might get **implied ordering** (e.g., male = 0, female = 1)
* Ordinal categories may be encoded arbitrarily unless you specify their order

Thus, we must use:

* 🟦 `OrdinalEncoder` for **ordinal features**
* 🟨 `OneHotEncoder` for **nominal features**

---

### 🔁 Automating the Process: `ColumnTransformer`

To apply different encoders to different columns **in a single preprocessing step**, we use:

```python
from sklearn.compose import ColumnTransformer
```

This allows us to:

* Select multiple columns
* Apply different transformers (e.g., `OrdinalEncoder`, `OneHotEncoder`) to each group
* Output a single transformed feature matrix ready for modeling

---

### 🧪 Example Use Case

Let’s say we have the following features:

| Column    | Type      | Example Values                | Encoding Strategy       |
| --------- | --------- | ----------------------------- | ----------------------- |
| Age       | Numerical | 22, 45, 31                    | ❌ Ignored for this demo |
| Gender    | Nominal   | Male, Female                  | ❌ Ignored for this demo |
| Review    | Ordinal   | Bad, Average, Good, Excellent | ✅ `OrdinalEncoder`      |
| Education | Ordinal   | HS, UG, PG                    | ✅ `OrdinalEncoder`      |

We'll build a preprocessing pipeline using `ColumnTransformer` that:

* Applies `OrdinalEncoder` to `'Review'` and `'Education'` (with defined order)
* Ignores `'Age'` and `'Gender'`

---

### ⚙️ Pipeline Setup (Conceptual)

```python
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Define order for ordinal features
ordinal_categories = [
    ['bad', 'average', 'good', 'excellent'],  # for 'Review'
    ['HS', 'UG', 'PG']                        # for 'Education'
]

# Create ColumnTransformer for ordinal columns only
preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=ordinal_categories), ['Review', 'Education'])
    ],
    remainder='drop'  # Drops other columns like Age and Gender
)
```

This pipeline transforms ordinal categorical features into numeric values **in one step**, making it ideal for clean and reproducible ML pipelines.

---

### ✅ Benefits of Using `ColumnTransformer`

| Advantage                         | Description                                                |
| --------------------------------- | ---------------------------------------------------------- |
| Modular                           | Separate logic for different column types                  |
| Scalable                          | Easily extend to pipelines with numeric + categorical data |
| Reproducible                      | Consistent encoding across train and test datasets         |
| Compatible with full ML pipelines | Use with `Pipeline`, `GridSearchCV`, etc.                  |

---

### 📍 Final Notes (from Transcript)

As mentioned in your transcript:

> "...For this, Scikit-learn provides a class called ColumnTransformer. We will specify that we have different types of data, and we will create separate pipelines for each. Using ColumnTransformer, we can apply all transformations in a single step..."

This clearly aligns with best practices in ML:

* Don’t manually encode each column separately
* Don’t mix up encoding strategies
* Instead, **use `ColumnTransformer` to handle both ordinal and nominal encodings** programmatically

---

In [157]:
import numpy as np
import pandas as pd

In [158]:
df = pd.read_csv('customer.csv')

In [159]:
df.sample(5)

,age,gender,review,education,purchased
39,76,Male,Poor,PG,No
17,22,Female,Poor,UG,Yes
30,73,Male,Average,UG,No
2,70,Female,Good,PG,No
37,94,Male,Average,PG,Yes


For this demo we will ignore Age (numerical) and gender (nomimal) columns.

To make it as a single step, skit learn has **ColumnTransformer** pipeline where you can apply **nomimal** and **ordinal** encoders in single step.

In [160]:
df = df.iloc[:,2:]

In [161]:
df.head()

,review,education,purchased
0,Average,School,No
1,Poor,UG,No
2,Good,PG,No
3,Good,PG,No
4,Average,UG,No


In [162]:
from sklearn.model_selection import train_test_split

# 🧾 We assume df is a pandas DataFrame with 3 or more columns
# For example:
#    Column 0: Review (categorical)
#    Column 1: Education (categorical)
#    Column 2: Purchase (target)

# ✂️ Step: Splitting into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, 0:2],     # ✅ Select all rows, and first two columns as features (X)
    df.iloc[:, -1],      # ✅ Select all rows, and last column as target (y)
    test_size=0.2,       # ⏳ Use 20% of the data for testing (80% for training)
    random_state=42      # 🧪 Ensures results are the same every time you run it
)

In [163]:
from sklearn.preprocessing import OrdinalEncoder

In [164]:
X_train

,review,education
12,Poor,School
4,Average,UG
37,Average,PG
8,Average,UG
3,Good,PG
6,Good,School
41,Good,PG
46,Poor,PG
47,Good,PG
15,Poor,UG


In [165]:
from sklearn.preprocessing import OrdinalEncoder

# 🎯 Goal: Convert ordinal categorical values to numbers
#    while preserving their meaningful order (ranking)

# 🧠 'categories' takes a list of lists
#     - Each sublist defines the correct order for one feature column

# Column 0: Review has 3 levels → Poor < Average < Good
# Column 1: Education has 3 levels → School < UG < PG

oe = OrdinalEncoder(categories=[
    ['Poor', 'Average', 'Good'],    # 👈 Order for Review column (e.g. product feedback)
    ['School', 'UG', 'PG']          # 👈 Order for Education column (lowest to highest)
])

In [166]:
# 🏗️ Fit the OrdinalEncoder on training features only
# This learns how to convert categories into numbers using the predefined order
oe.fit(X_train)

# ⚠️ Do NOT fit on X_test – that would leak information from test to train

OrdinalEncoder(categories=[['Poor', 'Average', 'Good'], ['School', 'UG', 'PG']])

In [167]:
# Save a before copy for later comparison
X_train_orig = X_train.copy()

In [168]:
X_train_t = oe.transform(X_train)
X_test_t  = oe.transform(X_test)

In [169]:
import pandas as pd

# Convert NumPy array to DataFrame using original column names
X_train_encoded = pd.DataFrame(X_train_t, columns=X_train.columns, index=X_train.index)

In [170]:
# Concatenate original and encoded values column-wise
X_compare = pd.concat([X_train_orig.reset_index(drop=True), X_train_encoded.reset_index(drop=True)], axis=1)
X_compare.columns = [f'{col} (orig)' for col in X_train.columns] + [f'{col} (encoded)' for col in X_train.columns]

X_compare.head(10)  # View first 10 rows

,review (orig),education (orig),review (encoded),education (encoded)
0,Poor,School,0.0,0.0
1,Average,UG,1.0,1.0
2,Average,PG,1.0,2.0
3,Average,UG,1.0,1.0
4,Good,PG,2.0,2.0
5,Good,School,2.0,0.0
6,Good,PG,2.0,2.0
7,Poor,PG,0.0,2.0
8,Good,PG,2.0,2.0
9,Poor,UG,0.0,1.0


In [171]:
# X_train.head(15)

In [172]:
# X_train_t

In [173]:
# import pandas as pd

# # Reuse the original column names from X_train
# X_train_encoded = pd.DataFrame(X_train_t, columns=X_train.columns, index=X_train.index)
# X_train_encoded.head(15)

In [174]:
oe.categories_

[array(['Poor', 'Average', 'Good'], dtype=object),
 array(['School', 'UG', 'PG'], dtype=object)]

## Use LabelEncoder for y

In [175]:
from sklearn.preprocessing import LabelEncoder

In [176]:
le = LabelEncoder()

---

### ⚠️ <u>Use `LabelEncoder` Only for Target (`y`), Not for Features (`X`)</u>

### 🔹 Summary Statement

> **Label Encoding must be applied exclusively to the target variable (`y`) in classification tasks. It should not be used on input features (`X`), especially when those features are categorical.**

---

### 🎯 Why This Is Important

#### ✔ Intended Use of `LabelEncoder`:

* The `LabelEncoder` class in `scikit-learn` is specifically designed for encoding **target labels** (i.e., the dependent variable `y`).
* It transforms string class labels (e.g., `'yes'`, `'no'`) into integer values (e.g., `1`, `0`).

#### ❌ Misuse on Input Features (`X`):

* Applying `LabelEncoder` to input features introduces an **arbitrary and misleading numeric order** between categories.
* For example:

  ```python
  le = LabelEncoder()
  le.fit_transform(['red', 'blue', 'green'])  # Might give [2, 0, 1]
  ```

  This falsely implies that `red > green > blue`, which is **incorrect** for nominal data.

---

### 📌 Recommended Alternatives for `X`

| Feature Type        | Use This Encoder | Notes                                   |
| ------------------- | ---------------- | --------------------------------------- |
| Ordinal Categorical | `OrdinalEncoder` | Specify order explicitly                |
| Nominal Categorical | `OneHotEncoder`  | No implied order; produces binary flags |

---

### ✅ Correct Usage Example

```python
from sklearn.preprocessing import LabelEncoder

# Target labels
y = ['yes', 'no', 'yes', 'no']

# Apply LabelEncoder ONLY to target
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # ✅ valid usage
```

---

### ❌ Incorrect Usage on Input

```python
from sklearn.preprocessing import LabelEncoder

X = pd.DataFrame({'color': ['red', 'blue', 'green']})

le = LabelEncoder()
X['color_encoded'] = le.fit_transform(X['color'])  # ❌ Not recommended
```

📛 **Problem**: `LabelEncoder` assigns arbitrary integers to strings → the model may infer a false ranking.

---

### 📘 Conclusion

> ✅ Use `LabelEncoder` only on `y` (target)
> ❌ Never use `LabelEncoder` on `X` (features) — use `OrdinalEncoder` or `OneHotEncoder` instead depending on feature type

---


In [177]:
from sklearn.preprocessing import LabelEncoder

# 🎯 Goal: Encode the target labels (y_train) for classification
# For example: ['yes', 'no', 'yes'] → [1, 0, 1]

le = LabelEncoder()      # ✅ Create LabelEncoder object
le.fit(y_train)          # ✅ Learn the label → integer mapping from training labels only

# ⚠️ Never use LabelEncoder on input features (X)

LabelEncoder()

In [178]:
# inspect the class-label mapping using
le.classes_

array(['No', 'Yes'], dtype=object)

In [179]:
# le.inverse_transform([1, 0]) ➝ ['yes', 'no']

In [180]:
# Store the original labels before encoding
y_train_orig = y_train.copy()
y_train_orig.head(15)

,purchased
12,No
4,No
37,Yes
8,No
3,No
6,No
41,Yes
46,No
47,Yes
15,No


In [181]:
# 🎯 Our target column (y) contains binary labels: 'yes' and 'no'
# Machine learning models like XGBoost require the target to be numeric (e.g., 0 and 1)

# 🧠 LabelEncoder will convert:
#     'no'  → 0
#     'yes' → 1
# It learns this mapping during le.fit(y_train), and now we're applying it to both splits.

# 🔁 Apply the learned encoding to training set
y_train = le.transform(y_train)

# 🔁 Apply the same encoding to test set
# 🚫 Do NOT fit again on y_test — that would break consistency
y_test = le.transform(y_test)

In [182]:
# Create a DataFrame for visual comparison
import pandas as pd

compare_df = pd.DataFrame({
    'Original Label': y_train_orig,
    'Encoded Label': y_train
})

compare_df.head(10)  # Show first 10 for quick inspection

,Original Label,Encoded Label
12,No,0
4,No,0
37,Yes,1
8,No,0
3,No,0
6,No,0
41,Yes,1
46,No,0
47,Yes,1
15,No,0
